<a href="https://colab.research.google.com/github/LuisManuelCatzoliSoriano/Simulaci-n_II/blob/main/19/11/26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Considere el modelo dado por la ecuación $dx_t=\mu x_tdt+\sigma
x_tdw_t$ con $x_0=10, \mu=0.002, \sigma=0.03$.

$a)$ Genere una trayectoria $x_0,x_1,...,x_50$ por simulacion de Montecarlo.

$b)$ Estime por simulación el valor de la función de verosimilitud para la trayectoria generada en $a)$.

$c)$ Use la técnica de variables antitéticas para reducir la varianza del valor obtenido en $b)$. Cuál fue el porcentaje de reducción de varianza?

In [1]:

import numpy as np

# Parámetros
x0 = 10
mu = 0.002
sigma = 0.03
dt = 1
n = 50
M = 10000

rng = np.random.default_rng(123)

# Función para generar una trayectoria
def trayectoria(z):
    x = np.zeros(n + 1)
    x[0] = x0

    for i in range(n):
        x[i+1] = x[i] * np.exp(
            (mu - sigma**2/2)*dt
            + sigma*np.sqrt(dt)*z[i]
        )

    return x

# Log-verosimilitud de una trayectoria
def log_verosimilitud(x):
    incrementos = np.diff(np.log(x))
    a = (mu - sigma**2/2)*dt

    return np.sum(
        -np.log(x[1:])
        - np.log(sigma*np.sqrt(2*np.pi*dt))
        - (incrementos-a)**2/(2*sigma**2*dt)
    )

# Inciso a: una trayectoria
z = rng.normal(0, 1, n)
x = trayectoria(z)

print("TRAYECTORIA:")
print(np.round(x, 6))

# Inciso b: verosimilitud
logL = log_verosimilitud(x)
L = np.exp(logL)

print("\nINCISO B")
print("Log-verosimilitud:", logL)
print("Verosimilitud:", L)

# Inciso c: simulaciones independientes y antitéticas
Z = rng.normal(0, 1, (M, n))

L_normal = np.empty(2*M)
L_antitetico = np.empty(M)

for j in range(M):
    xp = trayectoria(Z[j])
    xm = trayectoria(-Z[j])

    lp = log_verosimilitud(xp)
    lm = log_verosimilitud(xm)

    Lp = np.exp(lp)
    Lm = np.exp(lm)

    L_normal[2*j] = Lp
    L_normal[2*j+1] = Lm

    L_antitetico[j] = (Lp + Lm)/2

# Estimaciones
estimacion_MC = np.mean(L_normal)
estimacion_AV = np.mean(L_antitetico)

# Varianzas de los estimadores
var_MC = np.var(L_normal, ddof=1)/2
var_AV = np.var(L_antitetico, ddof=1)

reduccion = (1 - var_AV/var_MC)*100

print("\nINCISO C")
print("Estimación Monte Carlo:", estimacion_MC)
print("Estimación antitética:", estimacion_AV)
print("Varianza Monte Carlo:", var_MC)
print("Varianza antitética:", var_AV)
print("Reducción de varianza (%):", reduccion)


TRAYECTORIA:
[10.        9.722681  9.630911 10.025839 10.099994 10.398814 10.59683
 10.412541 10.599635 10.515724 10.430665 10.477342 10.024049 10.405163
 10.213592 10.541053 10.600669 11.116474 10.915448 10.830597 10.957874
 10.271614 10.546265 11.062607 11.460713 11.741364 11.70819  12.186097
 12.60466  12.773788 12.795566 12.67708  12.236689 12.714913 11.931289
 11.817835 11.89468  12.224446 12.907874 13.318942 13.223448 13.536442
 13.053906 13.647216 13.604373 13.352832 13.119593 13.115777 12.982241
 13.927003 13.651292]

INCISO B
Log-verosimilitud: -18.15554714347784
Verosimilitud: 1.3036051264082888e-08

INCISO C
Estimación Monte Carlo: 53.47313097571579
Estimación antitética: 53.47313097571579
Varianza Monte Carlo: 13027437.327493573
Varianza antitética: 13026658.935384158
Reducción de varianza (%): 0.005975020948834064
